# Project flow

This notebook shows the calculation order and final portfolio totals.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"
CFG = yaml.safe_load((ROOT / "config" / "project.yaml").read_text())

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

Calculation order

1. Load loan and macro history.
2. Define default and build model features.
3. Estimate 12-month PD and monthly hazard.
4. Assign stages and build macro scenarios.
5. Create monthly PD, LGD, EAD and discount-factor terms.
6. Calculate scenario ECL and apply scenario weights.
7. Reconcile loan, stage and portfolio totals.

In [2]:
query("select * from executive_summary")

,metric,value,unit
0,Reporting date,2026-03-01,date
1,Mortgage accounts,3471,count
2,Gross exposure,432963771.97,USD
3,Probability-weighted ECL,340767.410284789,USD
4,Portfolio coverage ratio,0.000787057560807654,percent
5,Performing exposure-weighted 12-month PD,0.00945910181717043,percent
6,ECL-driver-weighted performing LGD,0.0246376347131262,percent


In [3]:
query("select * from stage_summary order by stage")

,stage,loans,gross_exposure,ecl,coverage_ratio,exposure_share,ecl_share
0,1,241,"36,407,043.1800","1,387.6591",0.0000,0.0841,0.0041
1,2,3212,"393,794,048.4600","304,151.4746",0.0008,0.9095,0.8925
2,3,18,"2,762,680.3300","35,228.2766",0.0128,0.0064,0.1034


In [4]:
query("select * from scenario_summary order by scenario")

,scenario,scenario_weight,scenario_ecl,weighted_contribution
0,Base,0.5152,"337,567.2329","173,900.7725"
1,Downside,0.1672,"372,053.4097","62,221.1570"
2,Upside,0.3176,"329,484.2311","104,645.4807"


The detailed calculation is stored in ecl_projection_cube. Each row is one loan, one scenario and one future month.